In [ ]:
import sys
import subprocess

required = ["transformers", "datasets", "scipy", "pandas", "torch"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *required])


In [ ]:
import random
import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from scipy.stats import pearsonr, spearmanr

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

model_name = "sentence-transformers/all-MiniLM-L6-v2"
dataset_name = "glue"
dataset_config = "stsb"
split_name = "validation"
device = "mps" if torch.backends.mps.is_available() else "cpu"
batch_size = 256 if device == "mps" else 128
max_length = 128
start_time = time.time()

print({
    "model_name": model_name,
    "dataset": f"{dataset_name}/{dataset_config}",
    "split": split_name,
    "device": device,
    "batch_size": batch_size,
    "max_length": max_length,
    "seed": seed,
})


In [ ]:
ds = load_dataset(dataset_name, dataset_config, split=split_name)
df = ds.to_pandas()[["sentence1", "sentence2", "label"]].copy()
df["label"] = df["label"].astype(np.float32)

print({"num_examples": len(df), "columns": df.columns.tolist()})
print(df.head())


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.to(device)
model.eval()

print({
    "model_name": model_name,
    "hidden_size": int(model.config.hidden_size),
    "device": device,
})


In [ ]:
sentences1 = df["sentence1"].tolist()
sentences2 = df["sentence2"].tolist()

def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

def encode_texts(texts, batch_size=batch_size, max_length=max_length):
    all_embeddings = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i + batch_size]
            encoded = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt",
            )
            encoded = {k: v.to(device) for k, v in encoded.items()}
            outputs = model(**encoded)
            pooled = mean_pool(outputs.last_hidden_state, encoded["attention_mask"])
            pooled = F.normalize(pooled, p=2, dim=1)
            all_embeddings.append(pooled.detach().cpu())
    return torch.cat(all_embeddings, dim=0)

emb1 = encode_texts(sentences1)
emb2 = encode_texts(sentences2)

print({
    "emb1_shape": tuple(emb1.shape),
    "emb2_shape": tuple(emb2.shape),
    "dtype": str(emb1.dtype),
})


In [ ]:
cosine_similarity = (emb1 * emb2).sum(dim=1).numpy().astype(np.float32)
cosine_similarity = np.clip(cosine_similarity, -1.0, 1.0)
predicted_score_0_5 = ((cosine_similarity + 1.0) / 2.0) * 5.0
predicted_score_0_5 = np.clip(predicted_score_0_5, 0.0, 5.0).astype(np.float32)

preview_df = pd.DataFrame({
    "sentence1": df["sentence1"].head(10),
    "sentence2": df["sentence2"].head(10),
    "label": df["label"].head(10),
    "cosine_similarity": cosine_similarity[:10],
    "predicted_score_0_5": predicted_score_0_5[:10],
})
print(preview_df)


In [ ]:
labels = df["label"].to_numpy(dtype=np.float32)

pearson_cosine = pearsonr(cosine_similarity, labels).statistic
spearman_cosine = spearmanr(cosine_similarity, labels).statistic
pearson_rescaled = pearsonr(predicted_score_0_5, labels).statistic
spearman_rescaled = spearmanr(predicted_score_0_5, labels).statistic
mae_rescaled = float(np.mean(np.abs(predicted_score_0_5 - labels)))

results_df = df.copy()
results_df["cosine_similarity"] = cosine_similarity
results_df["predicted_score_0_5"] = predicted_score_0_5
results_df["abs_error"] = np.abs(results_df["predicted_score_0_5"] - results_df["label"])
results_df["distance_to_perfect_5"] = np.abs(5.0 - results_df["predicted_score_0_5"])

print(results_df[[
    "sentence1", "sentence2", "label", "cosine_similarity",
    "predicted_score_0_5", "abs_error"
]].head(10))


In [ ]:
nearest_to_perfect = results_df.sort_values(
    ["distance_to_perfect_5", "cosine_similarity"],
    ascending=[True, False]
).head(10)

lowest_similarity = results_df.sort_values("cosine_similarity", ascending=True).head(10)

print("nearest_to_perfect_score_examples")
print(nearest_to_perfect[[
    "sentence1", "sentence2", "label", "cosine_similarity",
    "predicted_score_0_5", "abs_error"
]].to_string(index=False))

print("lowest_similarity_examples")
print(lowest_similarity[[
    "sentence1", "sentence2", "label", "cosine_similarity",
    "predicted_score_0_5", "abs_error"
]].to_string(index=False))


In [ ]:
runtime_seconds = time.time() - start_time

print(f"device_used: {device}")
print(f"model_name: {model_name}")
print(f"dataset_split: {dataset_name}/{dataset_config}/{split_name}")
print(f"num_examples: {len(df)}")
print(f"pearson_cosine_similarity: {pearson_cosine:.6f}")
print(f"spearman_cosine_similarity: {spearman_cosine:.6f}")
print(f"pearson_rescaled_score_0_5: {pearson_rescaled:.6f}")
print(f"spearman_rescaled_score_0_5: {spearman_rescaled:.6f}")
print(f"mae_rescaled_score_0_5: {mae_rescaled:.6f}")
print(f"runtime_seconds: {runtime_seconds:.2f}")
